# MEAQ-T: DFT Output Parsing & Integration

This notebook shows how to:
- Parse VASP and Quantum ESPRESSO outputs
- Assess parsing confidence and diagnostics
- Extract key electronic structure properties

**Duration**: ~5 minutes

## Setup

In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from meaqt import parse_dft_report, parse_dft_text

print('✓ DFT parsers loaded')

## Load Benchmark Data

In [ ]:
# Path to benchmark fixtures
benchmark_dir = Path.cwd().parent / 'data' / 'benchmarks'

# Try to load a sample VASP output if available
vasp_sample_path = benchmark_dir / 'vasp_sample.out'
if vasp_sample_path.exists():
    with open(vasp_sample_path) as f:
        vasp_text = f.read()
    print(f"✓ Loaded VASP sample ({len(vasp_text)} chars)")
else:
    print("⚠ VASP sample not found; using mock data")
    vasp_text = """
    E-fermi :   -5.23156
    free  energy   TOTEN  =       -234.56789 eV
    band gap : 1.234
    """

## Parse VASP Output

In [ ]:
# Parse with auto-detection
vasp_report = parse_dft_report(vasp_text, code='auto')

print("VASP Parsing Result:")
print(f"  Code detected: {vasp_report.summary.code}")
print(f"  Fermi level: {vasp_report.summary.fermi_ev} eV")
print(f"  Total energy: {vasp_report.summary.total_energy_ev} eV")
print(f"  Band gap: {vasp_report.summary.band_gap_ev} eV")
print(f"  Confidence: {vasp_report.confidence:.1%}")
print(f"  Missing fields: {vasp_report.missing_fields}")
if vasp_report.notes:
    print(f"  Notes: {vasp_report.notes}")

## Parse QE Output (Quantum ESPRESSO)

In [ ]:
# Mock Quantum ESPRESSO output
qe_text = """
the fermi energy is -3.5678 ev
!    total energy      =       -543.21987 Ry
band gap : 0.876
"""

qe_report = parse_dft_report(qe_text, code='auto')

print("QE Parsing Result:")
print(f"  Code detected: {qe_report.summary.code}")
print(f"  Fermi level: {qe_report.summary.fermi_ev} eV")
print(f"  Total energy: {qe_report.summary.total_energy_ev:.4f} eV (converted from Ry)")
print(f"  Band gap: {qe_report.summary.band_gap_ev} eV")
print(f"  Confidence: {qe_report.confidence:.1%}")

## Batch Processing DFT Results

In [ ]:
# Example: batch parse multiple outputs
outputs = {
    'Hf-W alloy (VASP)': vasp_text,
    'Zr-Ta alloy (QE)': qe_text,
}

results = []
for name, text in outputs.items():
    report = parse_dft_report(text, code='auto')
    results.append({
        'system': name,
        'code': report.summary.code,
        'fermi_ev': report.summary.fermi_ev,
        'total_energy_ev': report.summary.total_energy_ev,
        'band_gap_ev': report.summary.band_gap_ev,
        'confidence': report.confidence,
    })

df = pd.DataFrame(results)
print("\nBatch Parse Results:")
print(df.to_string(index=False))

## Confidence Scoring

The parser assigns confidence based on which fields were successfully extracted:
- Fermi level found: +45%
- Total energy found: +45%
- Band gap found: +10%

Additional penalty if total energy is positive (suggests unit/parsing error).

In [ ]:
# Example: high confidence vs. low confidence
complete_output = """
E-fermi :   -2.5
free  energy   TOTEN  =       -100.0 eV
band gap : 2.0
"""

incomplete_output = """
E-fermi :   -2.5
"""

complete_report = parse_dft_report(complete_output)
incomplete_report = parse_dft_report(incomplete_output)

print(f"Complete output confidence: {complete_report.confidence:.1%}")
print(f"  Missing: {complete_report.missing_fields}")
print()
print(f"Incomplete output confidence: {incomplete_report.confidence:.1%}")
print(f"  Missing: {incomplete_report.missing_fields}")

## Next Steps

1. **Calibrate moire models** with DFT band structure (replace toy minibands).
2. **Validate Seebeck predictions** against measured DFT transport.
3. **Screen compositions** using first-principles results.
4. **Link to ML models** trained on DFT data for faster prediction.